# EO benchmark · Questions, expected solutions and evaluation plans

A read-only inspection notebook. **All 294 benchmark questions** are included: 265 mapped tasks and 29 scope-awareness controls. Expand a task to inspect its question, reference workflow, required checks, rubric and available GT.

**How to use**

1. Read the inventory below, then browse the evaluation-family sections.
2. Expand the task card and its reference/GT panels; search full task IDs with your browser's Find.
3. Run all cells to refresh from the current repository files. The saved views already contain the current inventory.
4. Use the optional filter at the end to search by ID, question, method, region or evaluation requirement.

**Interpretation**

- A reference solution is an illustrative workflow, not a unique answer key when the task permits alternatives.
- Stored GT values must match the relevant source snapshot and candidate inputs before scoring. `error` and `skipped` records are not usable answers.
- A runtime reference recipe defines how to recompute expected values; it does not supply numerical GT values or calibrated tolerances.
- A mapped strategy or named check does not imply the validator has been implemented or an evaluation run completed. Missing evidence remains unverified.

Sources: `benchmark_tasks.json`, `task_evaluation_strategies.json`, `rubrics_108q_v3.json`, and `ground_truths.json` in `data/tasks/`.


In [ ]:
import html
import json
from collections import Counter
from pathlib import Path


class BenchmarkInspector:
    """Read-only view of current benchmark questions, solutions and evaluation plans."""

    GROUPS = {
        "qualitative": "Scientific interpretation · rubric scoring",
        "numerical_answer": "Numerical final answers",
        "numerical_artifact": "Numerical artifact transformations",
        "set_answer": "Lists and sets",
        "named_entity": "Named entities",
        "categorical_identifier": "CRS and categorical identifiers",
        "download_contract": "Downloads and product requirements",
        "retrieval_validation": "Retrieved content and source verification",
        "artifact_metadata": "Artifact metadata",
        "agent_as_judge": "Artifact inspection · agent as judge",
        "hybrid": "Combined evaluation components",
        "scope_awareness": "Scope awareness · outside the 265-task evaluation mapping",
    }

    def __init__(self, root):
        self.root = Path(root)
        self.tasks = self.read("benchmark_tasks.json")["tasks"]
        self.strategies = {t["task_ID"]: t for t in self.read("task_evaluation_strategies.json")["tasks"]}
        self.rubrics = {t["task_ID"]: t for t in self.read("rubrics_108q_v3.json")}
        self.ground_truths = self.read("ground_truths.json")
        ids = [t["task_ID"] for t in self.tasks]
        assert len(ids) == len(set(ids)), "Duplicate benchmark task IDs"
        assert set(self.strategies).issubset(ids), "Strategy references unknown task"
        assert set(self.ground_truths).issubset(ids), "GT references unknown task"
        assert all(r["task_text"] == next(t["task_text"] for t in self.tasks if t["task_ID"] == key)
                   for key, r in self.rubrics.items()), "Stale rubric task text"

    def read(self, filename):
        return json.loads((self.root / "data/tasks" / filename).read_text())

    @staticmethod
    def esc(value):
        return html.escape(str(value))

    def group(self, task):
        strategy = self.strategies.get(task["task_ID"])
        if strategy:
            return strategy["evaluation_strategy"]
        return "scope_awareness" if "Scope Awareness" in task.get("task_labels", []) else "unmapped"

    def select(self, group=None, query="", gt_only=False):
        query = query.casefold().strip()
        return [t for t in self.tasks
                if (group is None or self.group(t) == group)
                and (not gt_only or t["task_ID"] in self.ground_truths)
                and (not query or query in json.dumps({
                    "task": t, "evaluation": self.strategies.get(t["task_ID"], {})},
                    ensure_ascii=False).casefold())]

    def table(self, headers, rows):
        return '<table><thead><tr>' + ''.join(f'<th>{self.esc(h)}</th>' for h in headers) + '</tr></thead><tbody>' + ''.join(
            '<tr>' + ''.join(f'<td>{self.esc(v)}</td>' for v in row) + '</tr>' for row in rows) + '</tbody></table>'

    def block(self, title, value):
        return f'<details class="sub"><summary>{self.esc(title)}</summary><pre>{self.esc(json.dumps(value, indent=2, ensure_ascii=False))}</pre></details>'

    def summary(self):
        counts = Counter(self.group(t) for t in self.tasks)
        rows = [[self.GROUPS.get(group, group), counts[group],
                 sum(t['task_ID'] in self.ground_truths for t in self.select(group)),
                 sum(t['task_ID'] in self.rubrics for t in self.select(group))]
                for group in self.GROUPS if counts[group]]
        status = Counter(entry.get('_status', 'stored reference') for entry in self.ground_truths.values())
        return self.style() + f'<div class="bench"><h2>Current inventory</h2><p><b>{len(self.tasks)} questions</b> · {len(self.strategies)} evaluation mappings · {len(self.rubrics)} rubrics · {len(self.ground_truths)} GT records</p>' + self.table(
            ['Primary evaluation family', 'Questions', 'GT records (including error/skipped)', 'Rubrics'], rows) + f'<p>GT record status: {self.esc(dict(status))}.</p><p>Each question appears once by its primary strategy. Methods can overlap; hybrid tasks may combine numeric, set, rubric and artifact checks.</p></div>'

    def card(self, task):
        key = task['task_ID']
        plan = self.strategies.get(key, {})
        methods = plan.get('evaluation_methods', [])
        gt = self.ground_truths.get(key)
        runtime = plan.get('numerical_evaluation', {})
        gt_label = ('No stored fixed GT' if gt is None else
                    'GT unavailable: ' + gt['_status'] if '_status' in gt else 'Stored GT reference')
        if runtime.get('status') == 'runtime_reference_defined':
            gt_label += ' · runtime recomputation recipe'
        badges = ''.join(f'<span class="badge">{self.esc(x)}</span>' for x in task.get('task_labels', []))
        body = f'<details class="task" id="{key}"><summary><b>{key}</b> — {self.esc(task["task_text"])}</summary><div class="content">{badges}<p class="question">{self.esc(task["task_text"])}</p>'
        body += '<h4>Required / expected solution</h4><p>' + self.esc(task.get('reference_solution_description') or 'No reference description recorded.') + '</p>'
        body += '<p class="hint">Reference workflows illustrate a valid approach. They do not restrict choices left open by the question.</p>'
        for index, reference in enumerate(task.get('reference_solutions', []), 1):
            calls = []
            for step in reference.get('steps', []):
                arguments = ', '.join(f'{k}={json.dumps(v, ensure_ascii=False)}' for k, v in step.get('arguments', {}).items())
                call = f'{step.get("function_name", "tool")}({arguments})'
                if step.get('comment'):
                    call += '\n    # ' + step['comment']
                calls.append(call)
            body += f'<details class="sub"><summary>Reference workflow {index} · {len(calls)} steps</summary><pre>' + self.esc('\n\n'.join(calls)) + '</pre></details>'
        body += '<h4>Evaluation requirements</h4>'
        if plan:
            body += '<p><b>Primary strategy:</b> ' + self.esc(plan['evaluation_strategy']) + '<br><b>Methods:</b> ' + self.esc(', '.join(methods)) + '</p>'
            for field in ['evaluation_target', 'reference_policy', 'coverage_status', 'evaluation_notes']:
                if plan.get(field):
                    body += '<p><b>' + self.esc(field.replace('_', ' ').capitalize()) + ':</b> ' + self.esc(plan[field]) + '</p>'
            for field in ['fixed_task_requirements', 'evaluation_components', 'spatial_reduction', 'download_parameter_basis']:
                if field in plan:
                    body += self.block(field.replace('_', ' ').capitalize(), plan[field])
            checks = plan.get('deterministic_checks', [])
            if checks:
                body += '<h4>Check definitions</h4>' + self.table(['Check', 'Expected result / requirement', 'Mode', 'Readiness'], [
                    [c.get('type', c.get('id')), json.dumps(c.get('expected'), ensure_ascii=False),
                     c.get('validation_mode', 'unspecified'), c.get('execution_readiness', c.get('status', 'unspecified'))] for c in checks])
            if runtime:
                body += self.block('Numerical reference recipe, units, tolerances and execution status', runtime)
        else:
            body += '<p>Scope-awareness control: no entry in the 265-task evaluation mapping. Inspect the reference refusal / expected tool behavior above.</p>'
        rubric = self.rubrics.get(key, {}).get('rubric', {})
        if rubric:
            body += '<h4>Written-answer rubric</h4>' + self.table(['Criterion', 'Weight', 'Requirement'], [
                [c['type'], c['weight'], c['description']] for c in rubric.get('criteria', [])])
        body += '<h4>Ground truth</h4><p>' + self.esc(gt_label) + '</p>'
        if gt is not None:
            body += self.block('Full GT answer / reference, provenance and evidence', gt)
        elif not runtime:
            body += '<p class="hint">Use the specified source/artifact checks or rubric. Absence of fixed GT is not automatically an evaluation gap.</p>'
        if plan:
            body += self.block('Full evaluation configuration', plan)
        return body + '</div></details>'

    def render(self, group=None, query='', gt_only=False):
        tasks = self.select(group, query, gt_only)
        return self.style() + '<div class="bench"><p>' + str(len(tasks)) + ' matching questions</p>' + ''.join(self.card(t) for t in tasks) + '</div>'

    @staticmethod
    def style():
        return """<style>
        .bench {font:15px/1.6 system-ui,sans-serif;color:#243044;background:#fff;padding:18px;max-width:1250px}
        .bench .task{border:1px solid #cbd5e1;border-radius:9px;margin:12px 0;background:#f8fafc}
        .bench summary{cursor:pointer;padding:12px;line-height:1.65;overflow-wrap:anywhere}
        .bench .content{padding:0 18px 18px;background:#fff}
        .bench .question{font-size:16px;border-left:4px solid #2563eb;padding:12px;background:#eff6ff}
        .bench .badge{display:inline-block;background:#e0e7ff;padding:2px 8px;border-radius:5px;margin:8px 5px 0 0;font-size:12px}
        .bench table{width:100%;border-collapse:collapse;margin:12px 0}
        .bench th,.bench td{text-align:left;vertical-align:top;padding:10px;border-bottom:1px solid #e2e8f0;overflow-wrap:anywhere}
        .bench th{background:#eef2f6}.bench h4{margin:20px 0 6px;color:#174e8f}
        .bench pre{white-space:pre-wrap;overflow-wrap:anywhere;background:#f1f5f9;padding:12px;font-size:12px;max-height:600px;overflow:auto}
        .bench .hint{color:#58677d;font-size:13px}.bench .sub{border:1px solid #e2e8f0;margin:8px 0;border-radius:6px}
        </style>"""


In [ ]:
from IPython.display import HTML, display

# Works when launched from the repository root, notebooks/, or a descendant.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/tasks/benchmark_tasks.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the agenticeoecosystem repository.")
inspector = BenchmarkInspector(ROOT)
display(HTML(inspector.summary()))


Primary evaluation family,Questions,GT records (including error/skipped),Rubrics
Scientific interpretation · rubric scoring,88,0,88
Numerical final answers,40,40,0
Numerical artifact transformations,2,0,0
Lists and sets,19,2,0
Named entities,2,0,0
CRS and categorical identifiers,2,0,0
Downloads and product requirements,27,0,0
Retrieved content and source verification,13,0,0
Artifact metadata,5,0,0
Artifact inspection · agent as judge,36,0,0


## Scientific interpretation · rubric scoring

**88 questions.** All applicable secondary metrics and evaluation components are included inside each task card.

In [ ]:
display(HTML(inspector.render(group='qualitative')))


Criterion,Weight,Requirement
scientific_correctness,0.5,"The answer scientifically justifies the chosen dataset for mapping fire effects over the selected area and period, accurately describing its relevant capabilities and limitations—for example, MODIS burned-area and burn-date information for broad coverage or Sentinel-2 NBR for burn-related spectral contrast when coverage is adequately tiled."
grounded_and_complete,0.3,"The answer identifies a defensible affected study area, describes the retrieved data's actual spatial and temporal coverage or evidence without inventing unsupported patterns, and explains how that evidence relates to the 2019–2020 bushfire narrative."
internal_consistency,0.2,"The answer's suitability rationale and conclusions follow from the data it says it retrieved, with no contradictions between the selected area, dataset capabilities, observed evidence, and claims about the bushfires."
Criterion,Weight,Requirement
scientific_correctness,0.5,"The answer faithfully represents the scientific and EO content of the selected narrative about air quality, without introducing false claims or presenting source statements as stronger conclusions than they support. Faithful quotations are judged as quotations, not as additional candidate assertions; no unrequested scientific commentary is required."
grounded_and_complete,0.3,"The answer identifies a retrieved EO Dashboard narrative relevant to air quality and delivers its full text or an explicitly accessible artifact containing that text, with completeness and fidelity checked against the actual retrieved source. A summary, abstract, title or unsupported retrieval claim alone does not meet the full-content request."
internal_consistency,0.2,"The narrative identity, delivered content and any retrieval-status statements agree with each other and with the evidence available to the judge. Additional interpretation is not required, and no particular narrative from the reference is mandatory when another valid source satisfies the task."
Criterion,Weight,Requirement
scientific_correctness,0.5,"The answer gives a scientifically defensible assessment of whether the Tiber is discernible in the requested true-color imagery and, if it is, accurately describes its flow direction rather than confusing flow direction with the river's visible alignment."
grounded_and_complete,0.3,"The answer makes an image-specific visibility judgment and, when applicable, describes the river's direction across the central-Rome Sentinel-2 RGB scene for January–March 2023 at 10 m resolution and less than 20% cloud cover."


## Numerical final answers

**40 questions.** All applicable secondary metrics and evaluation components are included inside each task card.

In [ ]:
display(HTML(inspector.render(group='numerical_answer')))


Benchmark task inspection view

## Numerical artifact transformations

**2 questions.** All applicable secondary metrics and evaluation components are included inside each task card.

In [ ]:
display(HTML(inspector.render(group='numerical_artifact')))


Benchmark task inspection view

## Lists and sets

**19 questions.** All applicable secondary metrics and evaluation components are included inside each task card.

In [ ]:
display(HTML(inspector.render(group='set_answer')))


Check,Expected result / requirement,Mode,Readiness
canonical_inventory_set_match,null,deterministic,requires_validator_and_evidence


## Named entities

**2 questions.** All applicable secondary metrics and evaluation components are included inside each task card.

In [ ]:
display(HTML(inspector.render(group='named_entity')))


Check,Expected result / requirement,Mode,Readiness
exact_api_identifier,"""2m_temperature""",deterministic,requires_validator_and_evidence


## CRS and categorical identifiers

**2 questions.** All applicable secondary metrics and evaluation components are included inside each task card.

In [ ]:
display(HTML(inspector.render(group='categorical_identifier')))


Check,Expected result / requirement,Mode,Readiness
canonicalized_crs_equivalence,"{""authority"": ""EPSG"", ""code"": 32637, ""name"": ""WGS 84 / UTM zone 37N""}",deterministic,requires_validator_and_evidence
Check,Expected result / requirement,Mode,Readiness
canonicalized_crs_equivalence,"{""authority"": ""EPSG"", ""code"": 32722, ""name"": ""WGS 84 / UTM zone 22S""}",deterministic,requires_validator_and_evidence
utm_zone_hemisphere_match,"{""zone"": 22, ""hemisphere"": ""S""}",deterministic,requires_validator_and_evidence


## Downloads and product requirements

**27 questions.** All applicable secondary metrics and evaluation components are included inside each task card.

In [ ]:
display(HTML(inspector.render(group='download_contract')))


Check,Expected result / requirement,Mode,Readiness
artifact_exists_nonempty,true,deterministic,requires_validator_and_evidence
artifact_readable,true,deterministic,requires_validator_and_evidence
acquisition_provenance,"""artifact identity agrees with recorded acquisition/request""",deterministic,requires_validator_and_evidence
requested_product,"""MTG FCI FDHSI""",deterministic,requires_validator_and_evidence
requested_period,"""2023-09-05""",deterministic,requires_validator_and_evidence
requested_region,"""central Europe""",conditional_deterministic,requires_explicit_resolved_constraints_and_validator
observation_count,"{""minimum"": 1, ""maximum"": 3}",deterministic,requires_validator_and_evidence
download_method,"""datastore""",deterministic,requires_validator_and_evidence
restricted_readback_coverage,"""readback/derived artifact restricted to requested region""",conditional_deterministic,requires_explicit_resolved_constraints_and_validator
Check,Expected result / requirement,Mode,Readiness


## Retrieved content and source verification

**13 questions.** All applicable secondary metrics and evaluation components are included inside each task card.

In [ ]:
display(HTML(inspector.render(group='retrieval_validation')))


Check,Expected result / requirement,Mode,Readiness
paper_identity,"""Valid identifiers agreeing with retrieved arXiv metadata, without duplicates.""",deterministic,requires_validator_and_evidence
publication_date,"""Initial publication date on or after 2022-01-01; use source evidence and record date semantics.""",deterministic,requires_validator_and_evidence
Check,Expected result / requirement,Mode,Readiness
selected_paper_identity,"""Downloaded paper matches a relevant paper from the actual search results; search rank is not scored.""",deterministic,requires_validator_and_evidence
fulltext_artifact,"""Downloaded full-text artifact exists and is readable.""",deterministic,requires_validator_and_evidence
fulltext_readback,"""Trace/content evidence shows full-content retrieval; downloading alone does not establish reading.""",deterministic,requires_validator_and_evidence
Check,Expected result / requirement,Mode,Readiness
paper_identity,"""Selected paper belongs to the actual search results.""",deterministic,requires_validator_and_evidence
abstract_identity,"""Retrieved abstract agrees with the selected paper.""",deterministic,requires_validator_and_evidence
Check,Expected result / requirement,Mode,Readiness


## Artifact metadata

**5 questions.** All applicable secondary metrics and evaluation components are included inside each task card.

In [ ]:
display(HTML(inspector.render(group='artifact_metadata')))


Check,Expected result / requirement,Mode,Readiness
download_contract,"""Readable Sentinel-2 true-colour artifact covering central Rome within March 2023, with acquisition provenance.""",conditional_deterministic,requires_explicit_resolved_constraints_and_validator
reported_metadata,"""Driver, CRS, size, band count, data type and extent match actual candidate artifact; accept equivalent CRS representations.""",deterministic,requires_validator_and_evidence
Check,Expected result / requirement,Mode,Readiness
download_contract,"""Readable Copernicus 30 m DEM covering Mount Etna with product provenance.""",conditional_deterministic,requires_explicit_resolved_constraints_and_validator
reported_metadata,"""Driver, CRS, pixel resolution, band count, data type and extent agree with actual file.""",deterministic,requires_validator_and_evidence
Check,Expected result / requirement,Mode,Readiness
download_contract,"""ERA5 monthly mean geopotential at 500 hPa for June 2022 over Trento, Italy and its surroundings; readable regional artifact with provenance and nonempty grid coverage.""",conditional_deterministic,requires_explicit_resolved_constraints_and_validator
reported_metadata,"""Reported variables/time dimensions match actual file; accept equivalent names such as time/valid_time.""",deterministic,requires_validator_and_evidence
Check,Expected result / requirement,Mode,Readiness
download_contract,"""ERA5 total precipitation for January 2021, with readable regional artifact/readback over Tigray, Ethiopia.""",conditional_deterministic,requires_explicit_resolved_constraints_and_validator


## Artifact inspection · agent as judge

**36 questions.** All applicable secondary metrics and evaluation components are included inside each task card.

In [ ]:
display(HTML(inspector.render(group='agent_as_judge')))


Check,Expected result / requirement,Mode,Readiness
download_contract,"""Sentinel-2 near Kafr el-Sheikh, April 2023, B8/B4 at 10m.""",conditional_deterministic,requires_explicit_resolved_constraints_and_validator
reprojection,"""Reproject to EPSG:32636 using bilinear resampling before NDVI; verify trace and actual raster georeferencing.""",conditional_deterministic,requires_explicit_resolved_constraints_and_validator
ndvi_computation,"""Saved NDVI agrees with (B8-B4)/(B8+B4) on the reprojected candidate inputs, preserving valid masks and handling zero denominators; record numerical tolerance.""",deterministic,requires_validator_and_evidence
Check,Expected result / requirement,Mode,Readiness
download_contract,"""Sentinel-2 central Rome June 2023, B8/B4 at 10m and catalogue cloud cover <20%; verify actual input provenance.""",conditional_deterministic,requires_explicit_resolved_constraints_and_validator
ndvi_computation,"""NDVI agrees with candidate B8/B4 inputs within recorded tolerance; mask/zero denominator handling.""",deterministic,requires_validator_and_evidence
static_map,"""Readable saved PNG representing computed NDVI with red-yellow-green colour progression; AAJ existing image preview, no separate VLM evaluator and no unrequested legend/title.""",conditional_deterministic,requires_explicit_resolved_constraints_and_validator
Check,Expected result / requirement,Mode,Readiness
download_contract,"""Sentinel-2 agricultural Lodi area within June–August 2023, B8/B4 at 10m; any suitable scene, not all three months required.""",conditional_deterministic,requires_explicit_resolved_constraints_and_validator
ndvi_computation,"""NDVI matches candidate B8/B4 inputs within recorded tolerance.""",deterministic,requires_validator_and_evidence


## Combined evaluation components

**31 questions.** All applicable secondary metrics and evaluation components are included inside each task card.

In [ ]:
display(HTML(inspector.render(group='hybrid')))


Check,Expected result / requirement,Mode,Readiness
narrative_grounding,"""Selected narrative and reported study area agree with actual retrieved source evidence.""",semantic_judgment,requires_candidate_source_evidence_and_judge
download_contract,"""Readable Sentinel-2 true-colour download with acquisition provenance covers narrative-derived farmland north of Cairo. No CO2 measurement or unrequested interpretation.""",conditional_deterministic,requires_explicit_resolved_constraints_and_validator
Check,Expected result / requirement,Mode,Readiness
narrative_grounding,"""Selected narrative and reported study area agree with actual retrieved source evidence.""",semantic_judgment,requires_candidate_source_evidence_and_judge
full_content_retrieval,"""Actual full flood-narrative content retrieved, not only title/snippet; geographic area/approximate region supported. No data download or scientific synthesis required.""",deterministic,requires_validator_and_evidence
Check,Expected result / requirement,Mode,Readiness
narrative_grounding,"""Selected narrative and reported study area agree with actual retrieved source evidence.""",semantic_judgment,requires_candidate_source_evidence_and_judge
download_contract,"""Readable Sentinel-1 SAR download/provenance covers a suitable narrative-supported representative Arctic area with acquisition context appropriate for sea-ice/open-water contrast. No classified map or unrequested contrast explanation.""",conditional_deterministic,requires_explicit_resolved_constraints_and_validator
Check,Expected result / requirement,Mode,Readiness
narrative_grounding,"""Selected narrative and reported study area agree with actual retrieved source evidence.""",semantic_judgment,requires_candidate_source_evidence_and_judge


## Scope awareness · outside the 265-task evaluation mapping

**29 questions.** All applicable secondary metrics and evaluation components are included inside each task card.

In [ ]:
display(HTML(inspector.render(group='scope_awareness')))


Benchmark task inspection view

## Optional interactive search

Requires `ipywidgets`; the complete views above work without it. Select a primary family, enter a search term and click **Show tasks**. GT-only includes stored skipped/error records, which retain their explicit status.


In [ ]:
try:
    import ipywidgets as widgets
except ImportError:
    print("ipywidgets is unavailable. Use the grouped task views above, or call inspector.render(query='600015').")
else:
    family = widgets.Dropdown(options=[("All families", None)] +
                              [(title, key) for key, title in inspector.GROUPS.items()],
                              description="Family:", layout=widgets.Layout(width="750px"))
    query = widgets.Text(description="Search:", placeholder="Task ID, question, region, metric…",
                         layout=widgets.Layout(width="750px"))
    gt_only = widgets.Checkbox(value=False, description="Only tasks with stored GT records")
    button = widgets.Button(description="Show tasks", button_style="primary")
    result = widgets.Output()

    def show_tasks(_):
        with result:
            result.clear_output(wait=True)
            display(HTML(inspector.render(group=family.value, query=query.value, gt_only=gt_only.value)))

    button.on_click(show_tasks)
    display(widgets.VBox([family, query, gt_only, button, result]))
